# SokoRadar: alerts aligned with the forecasting model

**Goal:** test the forecasting model's spike alert on real market prices, add the ALPS label as context, and add price ranges to the forecasts.

**The final design: one alert, one label, one range**
* **Spike alert (M8a):** is a rise above 15% more likely than usual? This is the one alert.
* **ALPS label:** is the price high for the season right now? (Normal, Stress, Alert, Crisis). Context only.
* **Price range:** an 80% range around every price forecast.

**Plan**
1. Load the forecasting model's backtest results (ARIMA price forecasts and M8a spike chances)
2. ALPS phases for all 1,222 county series
3. New seasonal alerts: who warned first?
4. Sudden jumps above 15%: who caught them?
5. Price ranges around the ARIMA forecasts

**Rules:** same series and years as the forecasting model (2015 to 2024, each year predicted only from earlier years); results judged on real market prices only.

## Step 1. Load the forecasting model's backtest results
Two saved backtest files: ARIMA (the price forecast) and M8a (the chance of a rise above 15%). We join them into one table and recalculate the model's results, to check we start from exactly the same numbers.

In [1]:
import json, numpy as np, pandas as pd
from pathlib import Path

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / 'output').exists())
OUT = ROOT / 'output'

def find_file(*names):
    """Look for the first of these file names in output/forecasts, then in output."""
    for folder in (OUT / 'forecasts', OUT):
        for n in names:
            if (folder / n).exists():
                return folder / n
    raise FileNotFoundError(f'None of {names} found in output or output/forecasts')

arima = pd.read_pickle(find_file('ARIMA_Daniella_s_orders.pkl', 'ARIMAorders.pkl'))
m8a = pd.read_pickle(find_file('M8a_full_balancing_weight.pkl'))

# one table: each row = one series, origin month and horizon, with both models' predictions
KEY = ['series', 'origin_month', 'horizon']
bt = (arima.rename(columns={'forecast_price': 'arima_price'})
           .merge(m8a[KEY + ['forecast_price']].rename(columns={'forecast_price': 'spike_prob'}), on=KEY))
for c in ['commodity', 'price_type', 'county']:
    bt[c] = bt[c].astype(str)

# alert rule: spike = rise above 15%; alert when the spike chance is above the model's threshold
THRESHOLD = {1: 0.08230874618305906, 2: 0.2025086377495205, 3: 0.35055914314124076}
bt['spike'] = bt['target_price'] / bt['origin_price'] - 1 > 0.15
bt['alert'] = bt['spike_prob'] > bt['horizon'].map(THRESHOLD)

print('ARIMA rows:', len(arima), '| M8a rows:', len(m8a), '| matched:', len(bt))
print('Series:', bt['series'].nunique(), '| years:', bt['fold year'].min(), 'to', bt['fold year'].max())
print('Answers recorded in a market:', f"{bt['target_observed'].mean():.1%}")

real = bt[bt['target_observed']]
check = pd.DataFrame({
    'ARIMA error %': (real['arima_price'] - real['target_price']).abs().div(real['target_price']).groupby(real['horizon']).mean() * 100,
    'spikes caught %': (real['alert'] & real['spike']).groupby(real['horizon']).sum() / real['spike'].groupby(real['horizon']).sum() * 100,
    'alerts right %': (real['alert'] & real['spike']).groupby(real['horizon']).sum() / real['alert'].groupby(real['horizon']).sum() * 100,
})
print(check.round(1))

StopIteration: 

**What the output shows**

1. Two backtest files (ARIMA price forecasts and M8a spike chances) match row for row: 432,588 rows, 1,222 county series, 2015 to 2024.
2. Only 6.8% of the answers were recorded in a market, so all scores use those real prices only.
3. The results match the team's model card exactly: ARIMA is off by 9.0%, 12.3% and 14.5% at 1, 2 and 3 months; M8a's alert catches 48%, 45% and 34% of spikes above 15%, and 22%, 32% and 39% of its alerts come true.
4. We are therefore building on the team's  results

## Step 2. ALPS phases for all 1,222 county series
For every series and month, how far the price sits above its normal level for that time of year, learned from earlier months only. Each month is labelled Normal, Stress, Alert or Crisis. The normal level is learned from all prices, and results are judged on real prices only.

In [ ]:
PHASES = ['Normal', 'Stress', 'Alert', 'Crisis']
long = pd.read_csv(OUT / 'SokoRadar_County_2006_2024_long.csv', parse_dates=['month'])
long['observed'] = long['flag'].str.startswith('Observed')
long['series'] = long['county'] + ' | ' + long['commodity'] + ' | ' + long['price_type']
P = long.pivot(index='month', columns='series', values='price_kes')       # one column per series

t = np.arange(len(P)); mon = P.index.month
X = np.column_stack([np.ones(len(P)), t] + [(mon == m).astype(float) for m in range(2, 13)])

def alps_all(y, min_obs=36):
    """ALPS score for every month of one series: trend + season fitted on earlier months only."""
    out = np.full(len(y), np.nan)
    for i in range(min_obs, len(y)):
        beta, *_ = np.linalg.lstsq(X[:i], y[:i], rcond=None)
        sd = (y[:i] - X[:i] @ beta).std(ddof=X.shape[1])
        out[i] = (y[i] - X[i] @ beta) / sd
    return out

ALPS = pd.DataFrame({s: alps_all(P[s].values) for s in P.columns}, index=P.index)
lab = ALPS.rename_axis(index='month', columns='series').stack().dropna().rename('alps').reset_index()
lab = lab.merge(long[['month', 'series', 'county', 'commodity', 'price_type', 'price_kes', 'observed']],
                on=['month', 'series'])
lab['phase'] = pd.cut(lab['alps'], [-np.inf, 0.25, 1, 2, np.inf], right=False, labels=PHASES)

real = lab[lab['observed']]
print('Series scored:', lab['series'].nunique(), '| month-rows:', len(lab), '| real month-rows:', len(real))
print('Alert or Crisis, all months:', f"{lab['phase'].isin(['Alert', 'Crisis']).mean():.1%}",
      '| real months only:', f"{real['phase'].isin(['Alert', 'Crisis']).mean():.1%}")
core = ['Dry Maize', 'Sorghum', 'Beans Rosecoco', 'Red Irish Potatoes']
tab = (real[real['commodity'].isin(core)].groupby('commodity')['phase'].value_counts(normalize=True)
       .unstack().reindex(columns=PHASES).mul(100).round(1))
print(tab)

Series scored: 1222 | month-rows: 234624 | real month-rows: 13672
Alert or Crisis, all months: 23.4% | real months only: 21.6%
phase               Normal  Stress  Alert  Crisis
commodity                                        
Beans Rosecoco        68.8    11.8    8.2    11.1
Dry Maize             73.2     9.9    7.9     9.0
Red Irish Potatoes    66.5    13.7   13.7     6.1
Sorghum               66.6    11.0   10.7    11.6


**What the output shows**

1. ALPS phases were calculated for all 1,222 county series: 234,624 scored months (the first 36 months of each series are used to learn its normal level). 13,672 of them have a real market price.
2. About 1 in 5 real months is in Alert or Crisis (21.6%), similar to the share with estimates included (23.4%).
3. Maize is the calmest (73% of real months Normal); potatoes and sorghum are under pressure most often, about 1 in 4 months in Alert or Crisis.
4. Limit: for beans, maize and sorghum there are as many Crisis months as Alert months, or more. The normal level and usual swing are learned mostly from smooth estimated prices, so real prices can look more extreme than they are. Results are judged on real months only, and this is stated as a limit.

## Step 3. New seasonal alerts: who warned first?
A new alert is a market moving into Alert or Crisis after 3 calm months, with a real price that month. We check whether M8a's spike alert or the ALPS Stress signal warned 1 to 3 months before, and how often each signal is on.

In [ ]:
A = ALPS.rename_axis(index='month', columns='series')                 # ALPS score: one row per month, one column per series
OBS = long.pivot(index='month', columns='series', values='observed').reindex_like(A).astype(bool)

hit = A >= 1                                                           # Alert or Crisis
stress = (A >= 0.25) & (A < 1)                                         # Stress: above normal, not yet Alert
calm_before = ~hit.shift(1, fill_value=False) & ~hit.shift(2, fill_value=False) & ~hit.shift(3, fill_value=False)
in_backtest = pd.Series((A.index >= '2015-04-01') & (A.index <= '2024-12-01'), index=A.index)
events = hit & calm_before & OBS                                       # new alerts with a real price that month
events = events.loc[in_backtest]                                       # 1 to 3 months before them are inside 2015-2024

# the spike alert (M8a) pivoted to one row per origin month (True = alert raised at that horizon)
def m8a_at(k):
    w = bt[bt['horizon'] == k].pivot(index='origin_month', columns='series', values='alert')
    return w.reindex(index=A.index, columns=A.columns).fillna(False).astype(bool)

# how many months ahead each signal warned about month m (0 = no warning in the 3 months before)
def months_of_warning(warned_k):
    lead = pd.DataFrame(0, index=A.index, columns=A.columns)
    for k in (1, 2, 3):
        lead = lead.mask(warned_k(k), k)                               # keep the earliest warning (largest k)
    return lead.loc[in_backtest]

signals = {
    "model's spike alert (M8a)": lambda k: m8a_at(k).shift(k, fill_value=False),   # alert at m-k, k months ahead
    "Stress rule (ALPS 0.25 to 1)": lambda k: stress.shift(k, fill_value=False),       # market in Stress at m-k
}
leads = {name: months_of_warning(f) for name, f in signals.items()}
leads["Either one"] = np.maximum(*leads.values())

rows = []
for name, lead in leads.items():
    v = lead.values[events.values]
    rows.append({'signal': name, 'new alerts': len(v), 'warned 1+ months early %': 100 * (v > 0).mean(),
                 'average months of warning (when warned)': v[v > 0].mean()})
pd.set_option('display.width', 200)
print(pd.DataFrame(rows).set_index('signal').round(1))

# how often each signal is "on" in 2015-2024 (the cost: more alarms)
on_m8a = (m8a_at(1) | m8a_at(2) | m8a_at(3)).loc['2015-01-01':'2024-12-01']
on_stress = stress.loc['2015-01-01':'2024-12-01']
print('\nShare of series-months the signal is on, 2015-2024:',
      f"M8a {on_m8a.values.mean():.1%} | Stress rule {on_stress.values.mean():.1%}")

                              new alerts  warned 1+ months early %  average months of warning (when warned)
signal                                                                                                     
model's spike alert (M8a)            347                      50.7                                      2.5
Stress rule (ALPS 0.25 to 1)         347                      63.4                                      2.1
Either one                           347                      84.1                                      2.5

Share of series-months the signal is on, 2015-2024: M8a 22.5% | Stress rule 14.5%


**What the output shows**

1. From 2015 to 2024 there were 347 new alerts on real prices (a market moving into Alert or Crisis after 3 calm months), across all county series.
2. The model spike alert (M8a) warned of 50.7% of them, on average 2.5 months ahead. The simple Stress rule (market already above normal but not yet in Alert) warned of 63.4%, about 2.1 months ahead.
3. Together they warned of 84.1%, far more than either alone, so the two signals catch different problems and complement each other.
4. Cost: M8a is on in 22.5% of series-months, the Stress rule in 14.5%.
5. Caveats: new alerts are defined with ALPS, which favours the Stress rule

## Step 4. Sudden jumps above 15%: who caught them?
The fair counterpart to Step 3, judged on M8a's own target. For every forecast with a real price later, we check whether each signal flagged the jump, how many flags came true, and how often each signal rings.

In [ ]:
# add the ALPS Stress signal at each origin month to the backtest rows
st = stress.stack().rename('stress_now').reset_index().rename(columns={'month': 'origin_month'})
b = bt.merge(st, on=['origin_month', 'series'], how='left')
b['stress_now'] = b['stress_now'].fillna(False).astype(bool)
b['either'] = b['alert'] | b['stress_now']

real = b[b['target_observed']]                       # judge only where the later price was recorded in a market
rows = []
for h, g in real.groupby('horizon'):
    for name, col in [("The Model Spike alert (M8a)", 'alert'), ('Stress rule', 'stress_now'), ('Either one', 'either')]:
        on = g[col]
        rows.append({'horizon': h, 'signal': name,
                     'spikes caught %': 100 * (on & g['spike']).sum() / g['spike'].sum(),
                     'alerts right %': 100 * (on & g['spike']).sum() / on.sum(),
                     'signal on %': 100 * on.mean()})
res = pd.DataFrame(rows).set_index(['horizon', 'signal']).round(1)
print('Spikes (rise above 15%) on real prices, 2015-2024:', int(real['spike'].sum()), 'across the 3 horizons')
print(res)

Spikes (rise above 15%) on real prices, 2015-2024: 4625 across the 3 horizons
                                     spikes caught %  alerts right %  signal on %
horizon signal                                                                   
1       The Model Spike alert (M8a)             47.8            22.2         23.0
        Stress rule                             11.3            10.5         11.5
        Either one                              53.8            18.2         31.6
2       The Model Spike alert (M8a)             44.5            31.7         23.4
        Stress rule                             10.9            15.6         11.6
        Either one                              50.4            25.9         32.4
3       The Model Spike alert (M8a)             34.4            38.6         18.0
        Stress rule                              8.8            15.5         11.4
        Either one                              40.5            29.3         27.8


**What the output shows**

1. Judged on the model's own target (a rise above 15%, 4,625 spikes on real prices), the M8a spike alert is clearly the better signal: it catches 47.8%, 44.5% and 34.4% of spikes at 1, 2 and 3 months, against 11.3%, 10.9% and 8.8% for the Stress rule.
2. Only 10.5% to 15.6% of Stress signals were followed by a spike, close to chance; about 22% to 39% of M8a alerts were.
3. Combining them adds only about 6 points of spikes caught, while the alarm rings in about 32% of months instead of 23% and fewer alerts come true (18.2% against 22.2% at 1 month). For spikes, M8a alone is the better choice.
4. With Step 3: the spike alert is the stronger alert for sudden jumps, so it stays the one alert. The ALPS Stress label warns of seasonal pressure well, but it is weak at predicting jumps, so it is shown as context, not as a second alert.

## Step 5. Price ranges around the ARIMA forecasts
From how far off past forecasts were, we put a range around each forecast and test how often real prices land inside, on years the range had not seen.

In [ ]:
LEARN_AT = 0.85                 # width learned at the 85th percentile of past misses (see note), aiming for about 80% inside
MIN_ROWS = 150                  # a crop needs at least this many past real misses for its own width, otherwise the overall one

real = bt[bt['target_observed']].copy()
real['miss'] = np.abs(np.log(real['target_price'] / real['arima_price']))   # how far off the forecast was, in % terms (log)

def width(learn, h, crop):
    """Range half-width (in log terms) for one horizon and crop, learned from past misses only."""
    L = learn[learn['horizon'] == h]
    Lc = L[L['commodity'] == crop]
    return (Lc if len(Lc) >= MIN_ROWS else L)['miss'].quantile(LEARN_AT)

# honest test: for each year 2017-2024, the widths are learned only from earlier years
parts = []
for y in range(2017, 2025):
    learn, test = real[real['fold year'] < y], real[real['fold year'] == y]
    for (h, crop), g in test.groupby(['horizon', 'commodity']):
        q = width(learn, h, crop)
        parts.append(pd.DataFrame({'horizon': h, 'commodity': crop, 'inside': g['miss'] <= q, 'width': np.exp(q) - 1}))
check = pd.concat(parts)

print('Real prices inside the range, on years it had not seen (2017-2024):')
print((check.groupby('horizon').agg(**{'inside %': ('inside', 'mean'), 'average width ±%': ('width', 'mean')}) * 100).round(1))
core = ['Dry Maize', 'Sorghum', 'Beans Rosecoco', 'Red Irish Potatoes']
print('\nInside %, by crop:')
print((check[check['commodity'].isin(core)].groupby(['commodity', 'horizon'])['inside'].mean() * 100).unstack().round(1))

# final widths to use from now on, learned from all 2015-2024 misses
FINAL = pd.DataFrame({h: {c: 100 * (np.exp(width(real, h, c)) - 1) for c in core} for h in (1, 2, 3)}).round(0)
FINAL.columns = ['1 month ±%', '2 months ±%', '3 months ±%']
print('\nFinal price ranges (forecast plus or minus):')
print(FINAL)

Real prices inside the range, on years it had not seen (2017-2024):
         inside %  average width ±%
horizon                            
1            78.1              15.7
2            80.4              23.8
3            81.9              29.8

Inside %, by crop:
horizon                1     2     3
commodity                           
Beans Rosecoco      74.6  77.5  78.5
Dry Maize           77.6  79.9  80.5
Red Irish Potatoes  79.9  80.5  80.5
Sorghum             76.7  80.0  81.2

Final price ranges (forecast plus or minus):
                    1 month ±%  2 months ±%  3 months ±%
Dry Maize                 15.0         22.0         28.0
Sorghum                   18.0         24.0         27.0
Beans Rosecoco            16.0         21.0         24.0
Red Irish Potatoes        35.0         51.0         64.0


**What the output shows**

1. Each ARIMA forecast now comes with a range. Tested on 2017 to 2024, with each year's range learned only from earlier years, real prices landed inside 78.1%, 80.4% and 81.9% of the time at 1, 2 and 3 months, which meets the project target of 75% to 85%.
2. The range widens the further ahead we look: on average about ±16% at 1 month, ±24% at 2 months and ±30% at 3 months.
3. Each crop has its own width. With one width for all crops, potatoes landed inside only about 47% of the time; with a width per crop, every core crop is between 75% and 81%.
4. Final ranges: maize ±15%, ±22%, ±28%; sorghum ±18%, ±24%, ±27%; beans ±16%, ±21%, ±24%; potatoes ±35%, ±51%, ±64% at 1, 2 and 3 months. Potato prices are much harder to predict.
5. Note: widths are learned at the 85th percentile of past misses, because learning at 80% gave only 72% to 76% coverage (recent years were more volatile). This choice was made after seeing the first result, so it is stated openly.

## Conclusion

**What was done**
The forecasting model's spike alert was tested on the same county price data and years as the model (1,222 county series, 2015 to 2024, each year predicted only from earlier years), with every result judged on real market prices only. The ALPS label and price ranges were added.

**The final design: one alert, one label, one range**
1. **Spike alert:** "a rise above 15% is about twice as likely as usual". This is the one alert.
2. **ALPS label:** Normal, Stress, Alert or Crisis, showing whether the price is high for the season right now. It gives context; it is not a second alert.
3. **Price range:** an 80% range around every price forecast.

**Results**
1. **Spike alert:** catches 48% of rises above 15% one month ahead, and 22% of its alerts are followed by a rise, about twice as often as by chance.
2. **New seasonal alerts:** the spike alert warned of 51% of markets moving into Alert, about 2 months ahead. The ALPS Stress label warned of 63%, which is why it is shown beside the alert as context.
3. **Why ALPS is not a second alert:** for sudden jumps, the Stress label caught only 11% at 1 month, close to chance. Adding it to the spike alert caught only about 6 more points while the alarm rang more often.
4. **Price ranges:** on years they had not seen (2017 to 2024), 78%, 80% and 82% of real prices fell inside at 1, 2 and 3 months, within the 75% to 85% target. Maize next month is about ±15%; potatoes about ±35%.

**Limits**
1. An alert means higher risk, not a certain spike; about half of spikes come without one.
2. Only about 7% of later prices were recorded in a market, so all scores rest on that share.
3. The ALPS normal level is learned mostly from estimated prices, so real prices can look more extreme than they are.
4. The range width was learned at the 85th percentile after a first test at 80% gave too little coverage. This choice is stated openly.
5. Not yet tested on 2025 to 2026.

**Next steps**
1. Run the alert and the price ranges on 2025 to 2026 prices, together with the forecasting model.
2. Show the spike alert, the ALPS label and the price ranges in the SokoRadar app.